# Restroom Icon Matching

In [5]:
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm import tqdm
import os
from PIL import Image
import torch.nn.functional as F

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [6]:
def sort_paths_by_number(path_list):
    """
    Sort based on the numerical values of the filenames in the path,
    assuming all filenames can be converted to integers.
    """
    def get_file_number(path):
        file_name = os.path.splitext(os.path.basename(path))[0]
        return int(file_name)

    path_list.sort(key=get_file_number)

In [7]:
# import clip-vit model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
%pip install --quiet git+https://github.com/openai/CLIP.git
import clip

  Preparing metadata (setup.py) ... done


In [8]:
def infer(img_dir: Path, model, preprocess, no_grad=True):
    """
    Compute L2‑normalized feature embeddings for a list of image file paths using the CLIP visual encoder.
    """
    with torch.set_grad_enabled(not no_grad):
        image_paths = list(img_dir.glob("*.png"))

        image_paths_str = [str(p) for p in image_paths]

        sort_paths_by_number(image_paths_str)

        img_paths = image_paths_str

        embeddings = []
        for path in tqdm(img_paths):
            img = Image.open(path)
            x = preprocess(img)

            # 精度16->32
            x = x.type(torch.float32).unsqueeze(0).to(DEVICE)

            emb = model.visual(x)
            embeddings.append(emb)
            
        embeddings = torch.cat(embeddings)
        embeddings = F.normalize(embeddings, p=2, dim=1)
    return embeddings

In [9]:
def match_images(QUERY_DIR: Path, NON_QUERY_DIR: Path, model, preprocess, same=False):
    """
    For each query image in BASE_DATA_DIR/query, find its best matching image
    in BASE_DATA_DIR/gallery by computing cosine similarity of CLIP embeddings,
    then return the match indices as a list.
    """

    # 賓語提前

    query_embeddings = infer(QUERY_DIR, model, preprocess)
    non_query_embeddings = infer(NON_QUERY_DIR, model, preprocess)
    distances = torch.mm(query_embeddings, non_query_embeddings.t())
    distances = (distances + 1.) / 2.

    # 在相同時確保不取自己
    if same:
        distances.diagonal().fill_(0)

    topk_dists, topk_idxs = torch.topk(distances, 11, dim=1)  # distances has shape (num_queries, num_non_queries)

    topk_dists, topk_idxs = topk_dists.cpu(), topk_idxs.cpu()

    matches_dists, matches_idxs = topk_dists[:, 0], topk_idxs[:, 0]
    # baseline 為 [:, 1]取第二相似的
    matches_dists = matches_dists.cpu().numpy()
    matches_idxs = matches_idxs.cpu().numpy()

    for i in range(len(matches_idxs)):
        matches_idxs[i]+=1

    return matches_idxs.tolist()

In [ ]:
import kagglehub

kagglehub.login()

In [10]:
"""Generate submission CSV for test set only"""
DATA_PATH = Path(kagglehub.competition_download("restroom-ioai-2025"))

DATA_PATH

100%|██████████| 40.9M/40.9M [00:00<00:00, 106MB/s] 

Extracting files...


PosixPath('/root/.cache/kagglehub/competitions/restroom-ioai-2025')

In [ ]:
# 訓練一下模型
import torch.nn as nn

chop_model, preprocess = clip.load("ViT-B/16", device=DEVICE)
chop_model = chop_model.float()

for param in chop_model.parameters():
    param.requires_grad = False

chop_model.visual.proj.requires_grad =True

TRAIN_PATH = DATA_PATH / "training_set" / "training_set"

def get_path(type: str, s: str):
    return TRAIN_PATH / type / s

optim = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, chop_model.parameters()), 
    lr=5e-6, 
)

l = nn.CrossEntropyLoss()

chop_model.train()
num_epochs = 9

for epoch in range(num_epochs):
    for s in ["female", "male"]:
        optim.zero_grad()
        crop = infer(get_path("crop", s), chop_model, preprocess, False)
        orig = infer(get_path("orig", s), chop_model, preprocess, False)
        loss = l(crop, orig)
        print(loss)
        loss.backward()
        optim.step()

chop_model.eval()

In [ ]:
orig_model, preprocess = clip.load("ViT-B/16", device=DEVICE)

orig_model = orig_model.float()


for param in orig_model.parameters():
    param.requires_grad = False

orig_model.visual.proj.requires_grad =True

TRAIN_PATH = DATA_PATH / "training_set" / "training_set"

def get_path(type: str, s: str):
    return TRAIN_PATH / type / s

optim = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, orig_model.parameters()), 
    lr=5e-6, 
)

l = nn.CrossEntropyLoss()

orig_model.train()
num_epochs = 0

for epoch in range(num_epochs):
    for s in ["orig"]:
        optim.zero_grad()
        crop = infer(get_path(s, "female"), orig_model, preprocess, False)
        orig = infer(get_path(s, "male"), orig_model, preprocess, False)
        loss = l(crop, orig)
        print(loss)
        loss.backward()
        optim.step()

orig_model.eval()

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 17.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.33 GiB is allocated by PyTorch, and 76.15 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# Process test set and get results
print("Processing test set...")

TESTSET_PATH = DATA_PATH / "test_set" / "test_set"

    
chop_to_whole = match_images(
    TESTSET_PATH / "query",
    TESTSET_PATH / "gallery",
    chop_model, preprocess
)

whole_to_other = match_images(
    TESTSET_PATH / "gallery",
    TESTSET_PATH / "gallery",
    orig_model, preprocess,
    same=True
)

print(f"chop_to_whole: {chop_to_whole}")
print(f"whole_to_other: {whole_to_other}")
test_results = [whole_to_other[i-1] for i in chop_to_whole]

# Create CSV in the required format
submission_df = pd.DataFrame({
    'id': [0],
    'array': [str(test_results)]
})

# Save to CSV
output_file = "restroom-2step_crop-model-epoch=9.csv"
submission_df.to_csv(output_file, index=False)

print(f"Submission file created successfully at {output_file}")
print(f"Results shape: {len(test_results)} matches")

Processing test set...


100%|██████████| 80/80 [00:01<00:00, 65.98it/s]

chop_to_whole: [1, 53, 55, 52, 51, 11, 30, 4, 34, 2, 33, 60, 25, 72, 41, 56, 40, 38, 28, 6, 23, 48, 42, 9, 5, 36, 3, 8, 24, 72, 77, 72, 67, 76, 64, 61, 69, 68, 66, 70]
whole_to_other: [37, 7, 20, 59, 54, 46, 2, 29, 69, 47, 49, 48, 53, 30, 24, 42, 52, 33, 56, 3, 60, 41, 50, 15, 44, 28, 32, 26, 8, 14, 36, 27, 18, 39, 38, 31, 1, 35, 34, 57, 22, 16, 9, 25, 55, 6, 10, 12, 11, 23, 72, 76, 13, 5, 45, 19, 40, 51, 4, 21, 65, 14, 64, 63, 61, 73, 74, 71, 75, 80, 68, 78, 66, 76, 69, 52, 62, 72, 76, 70]
Submission file created successfully at restroom-2step_crop-model.csv
Results shape: 40 matches


In [ ]:
submission_df['array'][0]

'[37, 13, 45, 76, 72, 49, 14, 59, 39, 7, 18, 21, 44, 78, 22, 19, 57, 35, 26, 46, 50, 12, 16, 69, 54, 31, 20, 29, 15, 78, 62, 78, 74, 52, 63, 65, 75, 71, 73, 80]'

In [ ]:
from google.colab import drive
import shutil
import os

# 1. 掛載 Google Drive 到 Colab 虛擬機
# 執行後會跳出視窗請求權限，請點選「連線到 Google 雲端硬碟」並允許
drive.mount('/content/drive')

# 2. 定義來源檔案路徑與 Google Drive 的目標儲存路徑
source_path = output_file
# 預設儲存於雲端硬碟的根目錄，您也可以自訂資料夾路徑如 '/content/drive/MyDrive/Colab Notebooks/'
target_path = '/content/drive/MyDrive/submit'

# 3. 檢查檔案是否存在並執行複製上傳
if os.path.exists(source_path):
    shutil.copy(source_path, target_path)
    print(f"🎉 上傳成功！")
else:
    print(f"❌ 找不到來源檔案 {source_path}，請確認先前的預測程式碼是否有成功產生 CSV 檔。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🎉 上傳成功！
